# Array coverage

Given a subject's functional visual field (FVF) radius, what fraction of the 180 icons in a trial's search
array fall within it of the scanpath - i.e. were plausibly inspected, whether or not they were ever fixated
directly? This reuses `pipeline.stage2_align.fixations_to_icons.fixations_to_icons()` against the **full**
icon set (not just targets), which is exactly the generalization it was built for.

In [ ]:
import numpy as np
import pandas as pd

import plotly.express as px
import plotly.io as pio

import constants as cnst
import config as cnfg
from analysis.helpers.read_data import load_data
from pipeline.stage2_align.fixations_to_icons import fixations_to_icons
from analysis.fvf.fvf import estimate_fvf

pio.renderers.default = 'notebook'      # 'notebook' or 'browser'

### Read data and estimate each subject's FVF

Uses estimator C (selection hazard) - see `compare_fvf_types.ipynb` for why A and B are unusable on this data.

In [ ]:
loaded_data = load_data(cnfg.OUTPUT_PATH)
fixations = loaded_data.fixations
icons = loaded_data.icons
metadata = loaded_data.metadata

fvf_table = estimate_fvf(loaded_data.fixation_target_dists)
fvf_by_subject = fvf_table['selection_hazard'].drop(index='all')
fvf_by_subject

### Fixation-to-icon distances, full array

One row per (fixation, icon) pair, for every one of the 180 icons per trial - not just targets.

This is ~180x the row count of the target-only table (`DataStore.fixation_target_dists`) - expect this cell to take noticeably longer to run across the full dataset.

In [ ]:
all_icon_dists = fixations_to_icons(fixations, icons, metadata)
len(all_icon_dists)

### Per-trial coverage

For each (subject, trial, icon): was any fixation within that subject's FVF radius? `array_coverage` is the
fraction of the trial's 180 icons for which the answer is yes.

In [ ]:
closest = (
    all_icon_dists
    .groupby([cnst.SUBJECT_STR, cnst.TRIAL_STR, cnst.ICON_STR], observed=True)[cnst.DISTANCE_DVA_STR]
    .min()
    .reset_index()
)
closest['fvf_radius'] = closest[cnst.SUBJECT_STR].map(fvf_by_subject)
closest = closest.dropna(subset=['fvf_radius'])  # subjects with too few opportunities for a hazard estimate
closest['within_fvf'] = closest[cnst.DISTANCE_DVA_STR] <= closest['fvf_radius']

array_coverage = (
    closest
    .groupby([cnst.SUBJECT_STR, cnst.TRIAL_STR], observed=True)['within_fvf']
    .agg(n_icons='size', n_covered='sum')
    .reset_index()
)
array_coverage['coverage_pct'] = 100 * array_coverage['n_covered'] / array_coverage['n_icons']
array_coverage.head()

### Distribution across trials and subjects

In [ ]:
fig = px.histogram(
    array_coverage, x='coverage_pct', color=cnst.SUBJECT_STR,
    nbins=40, opacity=0.6, template='plotly_white',
    title='Array coverage per trial (% of 180 icons within FVF of the scanpath)',
    labels={'coverage_pct': 'array coverage (%)'},
)
fig.show()

array_coverage['coverage_pct'].describe()